# Development Code Trial Archaeology
## Quantifying Hidden Clinical Trial History

---

### What this notebook does

1. Loads a collection, `collections/<COLLECTION>.csv` (set in section 1): INN, current code and legacy codes per row
2. Searches AACT for clinical trials by INN and by development code
3. Identifies "hidden" trials — those found only via a development code, not via the INN
4. Records which field, value and rule matched each trial (for verification)
5. Exports results with match evidence to `trial_archaeology/output/`

### Search Strategy

- **INN:** substring of `interventions.name`
- **Development codes:** the shared matching rule `patterns/aact_matching.py` — code as a token in five fields,
  ruled study-code patterns in `id_information.id_value`, code-scoped exclusions. Details and worked cases in section 4.

### Key Outputs

`trial_archaeology_<COLLECTION>_<timestamp>.xlsx`:
- **Summary sheet:** Per-drug trial counts (INN vs codes)
- **All_Trials sheet:** Every trial with match evidence
- **Hidden_Trials sheet:** Trials found ONLY via development codes
- **Residue sheet:** id_information values containing a code key, matched by no rule; counted = trial already in All_Trials for the same INN
- **Statistics sheet:** Overall metrics, collection, AACT last update, code fields, study-code patterns

`trials_verification_<COLLECTION>_<timestamp>.csv`: All_Trials as CSV

---

*Data source: AACT (Aggregate Analysis of ClinicalTrials.gov)*  
*Documentation: https://aact.ctti-clinicaltrials.org/schema*

## 1. Configuration

In [ ]:
# =============================================================================
# AACT DATABASE CREDENTIALS (REQUIRED)
# Get free account at: https://aact.ctti-clinicaltrials.org/users/sign_up
# =============================================================================

import os

from dotenv import dotenv_values

REPO_ROOT = "../.."

# AACT credentials from REPO_ROOT/.env (git-ignored, keys as in .env.example); values are never printed
ENV_FILE = f"{REPO_ROOT}/.env"
assert os.path.isfile(ENV_FILE), f"{ENV_FILE} not found: copy .env.example to .env and fill in the AACT keys"
env = dotenv_values(ENV_FILE)
missing = [k for k in ["AACT_USER", "AACT_PASSWORD"] if not env.get(k)]
assert not missing, f"{ENV_FILE} has no value for: {missing}"
AACT_USER = env["AACT_USER"]
AACT_PASSWORD = env["AACT_PASSWORD"]

AACT_HOST = "aact-db.ctti-clinicaltrials.org"
AACT_PORT = 5432
AACT_DATABASE = "aact"
AACT_SCHEMA = "ctgov"

COLLECTION = "fda_novel_2025"
COLLECTION_FILE = f"../../collections/{COLLECTION}.csv"

OUTPUT_DIR = f"{REPO_ROOT}/trial_archaeology/output"

# Matching rule: patterns/aact_matching.py (section 4); study-code patterns: patterns/study_code_patterns.csv
PATTERNS_FILE = f"{REPO_ROOT}/patterns/study_code_patterns.csv"

## 2. Setup

In [ ]:
import pandas as pd
import psycopg2
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore', message='.*pandas only supports SQLAlchemy.*')

import openpyxl  # Excel export; fails here if missing

print(f"Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# =============================================================================
# DATABASE CONNECTION
# =============================================================================

def get_connection():
    """Create a new database connection."""
    return psycopg2.connect(
        host=AACT_HOST, port=AACT_PORT, database=AACT_DATABASE,
        user=AACT_USER, password=AACT_PASSWORD,
        options=f"-c search_path={AACT_SCHEMA}"
    )

def run_query(query, params=None):
    """Execute a query and return results as DataFrame."""
    conn = get_connection()
    try:
        return pd.read_sql_query(query, conn, params=params)
    finally:
        conn.close()

# Test connection and show database stats (fails here if AACT is unreachable)
stats = run_query("""
    SELECT 
        (SELECT COUNT(*) FROM studies) as total_studies,
        (SELECT COUNT(*) FROM interventions) as total_interventions,
        (SELECT COUNT(*) FROM intervention_other_names) as total_other_names,
        (SELECT MAX(updated_at) FROM studies) as last_update
""")
AACT_LAST_UPDATE = str(stats['last_update'].iloc[0])
print(f"✓ Connected to AACT database")
print(f"  Studies:                  {stats['total_studies'].iloc[0]:,}")
print(f"  Interventions:            {stats['total_interventions'].iloc[0]:,}")
print(f"  Intervention other names: {stats['total_other_names'].iloc[0]:,}")
print(f"  Last update (studies):    {AACT_LAST_UPDATE}")

## 3. Drug Collection

Loaded from `collections/<COLLECTION>.csv`, columns `brand, inn, current_code, legacy_codes, sponsor`.
Provenance of each collection and its codes: `collections/README.md`.

**Code types:**
- **Current Code:** Sponsor's code at time of approval
- **Legacy Codes:** Earlier codes from originators, licensors, previous owners or partners

In [ ]:
# =============================================================================
# DRUG COLLECTION — loaded from collections/<COLLECTION>.csv (COLLECTION set in section 1)
# =============================================================================

drugs_df = pd.read_csv(COLLECTION_FILE, dtype=str, keep_default_na=False)
drugs_df = drugs_df.rename(columns={'brand': 'Brand', 'inn': 'INN', 'current_code': 'Current_Code',
                                    'legacy_codes': 'Legacy_Codes', 'sponsor': 'Sponsor'})
drugs_df = drugs_df[['Brand', 'INN', 'Current_Code', 'Legacy_Codes', 'Sponsor']]
drugs_df['Legacy_Codes'] = drugs_df['Legacy_Codes'].apply(lambda x: [c.strip() for c in x.split(';')] if x else [])
drugs_df['Legacy_Codes_Str'] = drugs_df['Legacy_Codes'].apply(lambda x: '; '.join(x) if x else '')

print(f"Collection: {COLLECTION}")
print(f"Loaded {len(drugs_df)} drugs")
print(f"  With current code: {(drugs_df['Current_Code'] != '').sum()}")
print(f"  With legacy codes: {(drugs_df['Legacy_Codes'].apply(len) > 0).sum()}")
print(f"  Total codes to search: {(drugs_df['Current_Code'] != '').sum() + drugs_df['Legacy_Codes'].apply(len).sum()}")

## 4. Matching Rule

One rule for code harvest and trial archaeology, defined in `patterns/aact_matching.py` and pasted below by
the builder (`trial_archaeology/tools/build_archaeology_notebook.py`). Do not edit the cell here.

- **Code as a token:** no letter or digit directly before or after the code, in every spelling
  (as given, no separator, space/hyphen swapped, prefix/number split). MK-3475 finds MK-3475-001 but not MK-3475A.
- **Study-code patterns:** ruled rows of `patterns/study_code_patterns.csv` find codes embedded in sponsor
  study codes, e.g. CLOU064A1301 for LOU064.
- **Exclusions:** code-scoped, for token collisions the rule cannot see (XM02-ONC-201 for ONC201).
- **Worked cases** are asserts at the end of the cell; the cell fails if the rule changes.

### Fields Searched

| Table | Field | Purpose |
|-------|-------|--------|
| `interventions` | `name` | Primary drug/intervention name (also INN search) |
| `intervention_other_names` | `name` | Aliases, synonyms, development codes |
| `studies` | `official_title`, `brief_title` | Trial titles (may contain code) |
| `id_information` | `id_value` | Sponsor study codes and secondary IDs |

In [ ]:
# =============================================================================
# AACT MATCHING RULE — one definition for code harvest and trial archaeology
# Source of truth: patterns/aact_matching.py (pasted into both notebooks by their builders)
# Rulings: see patterns/README.md. Worked cases are the asserts at the end.
# =============================================================================

import csv
import re

import pandas as pd

# Fields searched for codes (INN: interventions.name only)
CODE_FIELDS = [
    ("interventions.name",             "JOIN interventions f ON f.nct_id = s.nct_id",             "f.name"),
    ("intervention_other_names.name",  "JOIN intervention_other_names f ON f.nct_id = s.nct_id", "f.name"),
    ("studies.official_title",         "",                                                        "s.official_title"),
    ("studies.brief_title",            "",                                                        "s.brief_title"),
    ("id_information.id_value",        "JOIN id_information f ON f.nct_id = s.nct_id",            "f.id_value"),
]

# Code-scoped exclusions: (code key, exact matched value, lower-case) — token collisions the rule cannot see
EXCLUSIONS = {
    ("ONC201", "xm02-onc-201"),   # protocol ID of another sponsor (tbo-filgrastim, Teva), NCT02190721
}


def match_key(code):
    """Case, spaces, hyphens, slashes and brackets removed: MK-3475, MK 3475, MK3475 -> MK3475."""
    return re.sub(r"[\s\-/()+.]", "", code).upper()


def code_variants(code):
    """Spellings searched for one code: as given, no separator, space<->hyphen, prefix/number split."""
    c = code.lower().strip()
    no_sep = re.sub(r"[-\s]", "", c)
    variants = {c, no_sep}
    if " " in c:
        variants.add(c.replace(" ", "-"))
    if "-" in c:
        variants.add(c.replace("-", " "))
    m = re.match(r"^([a-z]+)(\d.*)$", no_sep)
    if m:
        variants.add(f"{m.group(1)}-{m.group(2)}")
        variants.add(f"{m.group(1)} {m.group(2)}")
    # Separator after the prefix dropped or swapped, number kept as given: "NNC 0113-0217" -> "nnc0113-0217"
    m = re.match(r"^([a-z]+)[-\s]?(\d.*)$", c)
    if m:
        for sep in ["", "-", " "]:
            variants.add(f"{m.group(1)}{sep}{m.group(2)}")
    return sorted(variants)


def _escape(s):
    return re.sub(r"([.^$*+?()\[\]{}|\\])", r"\\\1", s)


def token_regex(*codes):
    """Code(s) as a token: no letter or digit directly before or after. Same syntax in Python re and PostgreSQL ~*.
    Several spellings of one code (harvest candidates) give one regex with all their variants."""
    variants = sorted({v for c in codes for v in code_variants(c)})
    return r"(^|[^a-z0-9])(" + "|".join(_escape(v) for v in variants) + r")($|[^a-z0-9])"


def pattern_code_regex(code):
    """Code inside a study-code pattern: optional space/hyphen between letter prefix and number (lou[- ]?064)."""
    k = match_key(code).lower()
    m = re.match(r"^([a-z]+)(\d.*)$", k)
    assert m, f"Pattern codes need a letter prefix and a number: {code}"
    return f"{m.group(1)}[- ]?{_escape(m.group(2))}"


def load_patterns(path):
    """Ruled rows of patterns/study_code_patterns.csv; rows with status 'proposed' are not used."""
    with open(path, encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r["status"].startswith("ruled")]
    for r in rows:
        assert "{code}" in r["regex"], f"{r['pattern_id']}: regex lacks {{code}}"
        assert r["field"] == "id_information.id_value", f"{r['pattern_id']}: only id_information is supported"
    return rows


def pattern_regexes(code, patterns):
    """(pattern_id, regex) for every ruled pattern that lists this code."""
    k = match_key(code)
    return [(p["pattern_id"], p["regex"].replace("{code}", pattern_code_regex(code)))
            for p in patterns if k in {match_key(c) for c in p["applies_to_codes"].split(";")}]


def is_excluded(code, matched_value):
    return (match_key(code), str(matched_value).strip().lower()) in EXCLUSIONS


def trials_for_code(run_query, code, patterns):
    """All matches for one code: token search in CODE_FIELDS plus ruled study-code patterns.
    Returns DataFrame nct_id, matched_field, matched_value, search_term, rule."""
    rx = token_regex(code)
    parts = [run_query(f"""SELECT DISTINCT s.nct_id, '{label}' AS matched_field, {col} AS matched_value,
                               %s AS search_term, 'token' AS rule
                           FROM studies s {join} WHERE {col} ~* %s""", (code, rx))
             for label, join, col in CODE_FIELDS]
    for pid, prx in pattern_regexes(code, patterns):
        parts.append(run_query("""SELECT DISTINCT f.nct_id, 'id_information.id_value' AS matched_field,
                                      f.id_value AS matched_value, %s AS search_term, %s AS rule
                                  FROM id_information f WHERE f.id_value ~* %s""", (code, pid, prx)))
    found = [p for p in parts if len(p)]
    df = pd.concat(found, ignore_index=True) if found else parts[0]
    return df[~df["matched_value"].map(lambda v: is_excluded(code, v))].reset_index(drop=True)


def trials_for_inn(run_query, inn):
    """INN as a substring of interventions.name (full INN; combinations are split upstream into components)."""
    return run_query("""SELECT DISTINCT s.nct_id, 'interventions.name' AS matched_field, f.name AS matched_value,
                            %s AS search_term, 'inn' AS rule
                        FROM studies s JOIN interventions f ON f.nct_id = s.nct_id
                        WHERE LOWER(f.name) LIKE %s""", (inn.lower(), f"%{inn.lower()}%"))


def residue(run_query, code, patterns):
    """id_information values containing the code key (separators ignored) that neither the token rule nor a
    ruled pattern matches — for human review, never fed back automatically."""
    k = match_key(code).lower()
    rxs = [token_regex(code)] + [r for _, r in pattern_regexes(code, patterns)]
    df = run_query("""SELECT DISTINCT nct_id, id_value FROM id_information
                      WHERE regexp_replace(LOWER(id_value), '[^a-z0-9]', '', 'g') LIKE %s""", (f"%{k}%",))
    keep = ~df["id_value"].map(lambda v: any(re.search(r, v, re.I) for r in rxs) or is_excluded(code, v))
    return df[keep].reset_index(drop=True)


# -----------------------------------------------------------------------------
# Worked cases (AACT, 2026-10-15). A new review finding becomes a new line here.
# -----------------------------------------------------------------------------
def _tok(code, value):
    return bool(re.search(token_regex(code), value, re.I)) and not is_excluded(code, value)


_P = [{"pattern_id": "P1", "applies_to_codes": "LOU064; AIN457", "regex": "(^|[^a-z0-9])c{code}[a-z]",
       "field": "id_information.id_value", "status": "ruled"},
      {"pattern_id": "P2", "applies_to_codes": "CNTO1959", "regex": "(^|[^a-z0-9]){code}[a-z]{3,5}[0-9]{4}",
       "field": "id_information.id_value", "status": "ruled"}]


def _pat(code, value):
    return any(re.search(r, value, re.I) for _, r in pattern_regexes(code, _P))


# Format variants
assert _tok("BI 1015550", "BI1015550") and _tok("BI 1015550", "BI 1015550 (Nerandomilast)")
assert _tok("BMS 562247-01", "BMS-562247-01") and _tok("LY3298176", "LY-3298176")
assert _tok("NNC 0113-0217", "NNC0113-0217") and _tok("NNC 0113-0217", "NNC 0113-0217")   # titles NCT02016911, NCT02070510
# Short codes inside other tokens
assert not _tok("SS-31", "COMPASS-31") and not _tok("SS-31", "COMPASS31") and not _tok("SS-31", "CAPSS-314")
assert not _tok("WS-12", "AEWS1221") and _tok("SS-31", "SS-31")
assert not _tok("SB27", "compass27")
assert not _tok("TIC10", "HTIC-10-22") and not _tok("TIC10", "STIC/10/03")
assert not _tok("AB-106", "Sobi.EMAPALUMAB-106") and not _tok("AB-106", "CLI-05993AB1-06")
assert _tok("AB-106", "AB-106-C110")
# Digit after the code: a different number
assert not _tok("ONC201", "ONC-2010-001") and not _tok("ONC201", "R1979-ONC-20103")
assert not _tok("GSK572", "GSK572016")
# Code-scoped exclusion
assert not _tok("ONC201", "XM02-ONC-201") and _tok("ONC201", "ONC201-108")
# Letter after the code: another product or form (MK-3475A = s.c. co-formulation, review: other_product)
assert not _tok("MK-3475", "MK-3475A") and not _tok("MK-3475", "MK-3475A-C18")
assert not _tok("ONO-4538", "ONO-4538HSC")   # s.c. nivolumab + hyaluronidase (= BMS-986298, review: other_product)
assert not _tok("GSK1349572", "GSK1349572A")
# Separator then study number: the moiety's own study codes, also on partners' records
assert _tok("MK-3475", "MK-3475-001") and _tok("MK-3475", "KEYNOTE-C98, MK-3475-C98") and _tok("MK-3475", "MK-3475-A17")
assert _tok("NN9535", "NN9535-1820") and _tok("ONO-4538", "ONO-4538-24/CA209-473")
# Study-code patterns (P1, P2)
assert not _tok("LOU064", "CLOU064A1301") and _pat("LOU064", "CLOU064A1301") and _pat("LOU064", "CLOU064C12301")
assert _pat("AIN457", "CAIN457F2302/E1") and not _pat("AIN457", "MAIN457A_FVMBO009")   # residue, not a pattern
assert not _tok("CNTO1959", "CNTO1959PAP4001") and _pat("CNTO1959", "CNTO1959PAP4001")
assert not _pat("MK-3475", "MK-3475A-C18")   # no pattern row for MK-3475

In [ ]:
PATTERNS = load_patterns(PATTERNS_FILE)
print(f"Study-code patterns in use: {[p['pattern_id'] for p in PATTERNS]}")

In [ ]:
# =============================================================================
# SEARCH FUNCTIONS
# =============================================================================

def get_trial_details(nct_ids):
    """
    Fetch trial metadata for a set of NCT IDs.
    
    Returns: phase, start_date, overall_status, enrollment, brief_title
    """
    if not nct_ids:
        return pd.DataFrame()
    nct_list = list(nct_ids)
    placeholders = ','.join(['%s'] * len(nct_list))
    query = f"""
        SELECT s.nct_id, s.phase, s.start_date, s.overall_status,
               s.enrollment, s.brief_title, s.official_title
        FROM studies s
        WHERE s.nct_id IN ({placeholders})
        ORDER BY s.start_date NULLS LAST
    """
    return run_query(query, tuple(nct_list))


print("✓ Search functions defined")
print(f"  Code fields: {[label for label, _, _ in CODE_FIELDS]}")

## 5. Run Analysis

This cell iterates through all drugs and searches for trials.

**Output columns:**
- **INN:** Trials found by generic name
- **Curr:** Trials found by current development code
- **Leg:** Trials found by legacy codes
- **Hidden:** Trials found ONLY by codes (not INN)

In [ ]:
# =============================================================================
# MAIN ANALYSIS LOOP
# =============================================================================

print("Searching AACT database...")
print("=" * 90)
print(f"{'Drug':<28} {'INN':<6} {'Curr':<6} {'Leg':<6} {'Hidden':<8} Status")
print("=" * 90)

results = []
all_matches = []

for idx, drug in drugs_df.iterrows():
    brand = drug['Brand']
    inn = drug['INN']
    current_code = drug['Current_Code']
    legacy_codes = drug['Legacy_Codes']
    
    # -------------------------------------------------------------------------
    # 1. INN Search
    # -------------------------------------------------------------------------
    inn_matches = trials_for_inn(run_query, inn)
    inn_matches['search_type'] = 'INN'
    inn_matches['Brand'] = brand
    inn_matches['INN'] = inn
    all_matches.append(inn_matches)
    inn_ncts = set(inn_matches['nct_id'].tolist()) if len(inn_matches) > 0 else set()
    
    # -------------------------------------------------------------------------
    # 2. Current Code Search
    # -------------------------------------------------------------------------
    current_codes = [c.strip() for c in current_code.split(';') if c.strip()]
    current_matches = pd.concat([trials_for_code(run_query, c, PATTERNS) for c in current_codes],
                                ignore_index=True) if current_codes else pd.DataFrame()
    if len(current_matches) > 0:
        current_matches['code_label'] = current_code
        current_matches['search_type'] = 'Current_Code'
        current_matches['Brand'] = brand
        current_matches['INN'] = inn
        all_matches.append(current_matches)
    current_ncts = set(current_matches['nct_id'].tolist()) if len(current_matches) > 0 else set()
    
    # -------------------------------------------------------------------------
    # 3. Legacy Code Search
    # -------------------------------------------------------------------------
    legacy_ncts = set()
    for lc in legacy_codes:
        lc_matches = trials_for_code(run_query, lc, PATTERNS)
        if len(lc_matches) > 0:
            lc_matches['code_label'] = lc
            lc_matches['search_type'] = 'Legacy_Code'
            lc_matches['Brand'] = brand
            lc_matches['INN'] = inn
            all_matches.append(lc_matches)
            legacy_ncts.update(lc_matches['nct_id'].tolist())
    
    # -------------------------------------------------------------------------
    # Calculate hidden trials (found by code but NOT by INN)
    # -------------------------------------------------------------------------
    hidden_current = current_ncts - inn_ncts
    hidden_legacy = legacy_ncts - inn_ncts
    all_ncts = inn_ncts | current_ncts | legacy_ncts
    total_hidden = len(all_ncts - inn_ncts)   # union: a trial found by both current and legacy code counts once
    
    results.append({
        'Brand': brand,
        'INN': inn,
        'Current_Code': current_code,
        'Legacy_Codes': '; '.join(legacy_codes) if legacy_codes else '',
        'Trials_via_INN': len(inn_ncts),
        'Trials_via_Current': len(current_ncts),
        'Trials_via_Legacy': len(legacy_ncts),
        'Hidden_via_Current': len(hidden_current),
        'Hidden_via_Legacy': len(hidden_legacy),
        'Total_Hidden': total_hidden,
        'Total_Unique': len(all_ncts),
        'INN_NCTs': inn_ncts,
        'Current_NCTs': current_ncts,
        'Legacy_NCTs': legacy_ncts,
    })
    
    status = f"⚠️ {total_hidden} hidden" if total_hidden > 0 else "✓"
    print(f"{brand:<28} {len(inn_ncts):<6} {len(current_ncts):<6} {len(legacy_ncts):<6} {total_hidden:<8} {status}")

all_matches_df = pd.concat(all_matches, ignore_index=True) if all_matches else pd.DataFrame()
print("=" * 90)
print(f"\n✓ Analysis complete. Total match records: {len(all_matches_df):,}")

In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith('_NCTs')} for r in results])

total_unique = results_df['Total_Unique'].sum()
total_inn = results_df['Trials_via_INN'].sum()
total_hidden_current = results_df['Hidden_via_Current'].sum()
total_hidden_legacy = results_df['Hidden_via_Legacy'].sum()
total_hidden = results_df['Total_Hidden'].sum()   # per-drug union, no double count

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total unique trials found:     {total_unique:,}")
print(f"Trials via INN:                {total_inn:,}")
print(f"Hidden via current code:       {total_hidden_current:,} ({total_hidden_current/total_unique*100:.1f}%)")
print(f"Hidden via legacy code:        {total_hidden_legacy:,} ({total_hidden_legacy/total_unique*100:.1f}%)")
print(f"Total hidden:                  {total_hidden:,} ({total_hidden/total_unique*100:.1f}%)")
print(f"\nDrugs with hidden trials:      {(results_df['Total_Hidden'] > 0).sum()} of {len(results_df)}")

## 5b. Residue — id_information values for human review

`id_information` values that contain a code key (separators ignored) but that neither the token rule nor a
ruled study-code pattern matches. Candidates for a new pattern or exclusion under the admission rule in
`patterns/README.md`. Never fed back into the counts.

In [ ]:
residue_rows = []
for _, drug in drugs_df.iterrows():
    codes = [c.strip() for c in drug['Current_Code'].split(';') if c.strip()] + list(drug['Legacy_Codes'])
    for code in codes:
        r = residue(run_query, code, PATTERNS)
        if len(r) > 0:
            r.insert(0, 'code', code)
            r.insert(0, 'INN', drug['INN'])
            r.insert(0, 'Brand', drug['Brand'])
            residue_rows.append(r)

residue_df = (pd.concat(residue_rows, ignore_index=True) if residue_rows
              else pd.DataFrame(columns=['Brand', 'INN', 'code', 'nct_id', 'id_value']))
residue_df = residue_df.sort_values(['Brand', 'INN', 'code', 'nct_id']).reset_index(drop=True)
print(f"Residue: {len(residue_df)} id_information values, {residue_df['code'].nunique()} codes, "
      f"{residue_df['nct_id'].nunique()} trials")
display(residue_df.head(20))

## 6. Build Verification Table

Creates a per-trial table showing exactly what matched where.

**Purpose:** Enable human review for false positives.

**Columns:**
- `Found_via_*`: Boolean flags for how the trial was found
- `*_Match_Field`: Which database field matched
- `*_Match_Term`: Which search term matched
- `*_Match_Value`: The actual text that matched (for verification)

In [ ]:
# =============================================================================
# BUILD PER-TRIAL TABLE WITH MATCH EVIDENCE
# =============================================================================

trial_records = []

for r in results:
    brand, inn = r['Brand'], r['INN']
    current_code, legacy_codes = r['Current_Code'], r['Legacy_Codes']
    inn_ncts, current_ncts, legacy_ncts = r['INN_NCTs'], r['Current_NCTs'], r['Legacy_NCTs']
    all_ncts = inn_ncts | current_ncts | legacy_ncts
    
    for nct_id in all_ncts:
        trial_records.append({
            'nct_id': nct_id,
            'Brand': brand,
            'INN': inn,
            'Current_Code': current_code,
            'Legacy_Codes': legacy_codes,
            'Found_via_INN': nct_id in inn_ncts,
            'Found_via_Current': nct_id in current_ncts,
            'Found_via_Legacy': nct_id in legacy_ncts,
        })

trials_df = pd.DataFrame(trial_records)

# Add match detail columns
def get_match_info(nct_id, search_type):
    """Get match details for a trial and search type."""
    matches = all_matches_df[(all_matches_df['nct_id'] == nct_id) & 
                              (all_matches_df['search_type'] == search_type)]
    if len(matches) == 0:
        return '', '', ''
    fields = '; '.join(matches['matched_field'].unique())
    terms = '; '.join(matches['search_term'].unique())
    vals = '; '.join([str(v)[:60] for v in matches['matched_value'].unique()[:2]])
    return fields, terms, vals

print("Adding match details...")
for stype in ['INN', 'Current_Code', 'Legacy_Code']:
    key = stype.split('_')[0] if '_' in stype else stype
    fields, terms, vals = [], [], []
    for nct in trials_df['nct_id']:
        f, t, v = get_match_info(nct, stype)
        fields.append(f)
        terms.append(t)
        vals.append(v)
    trials_df[f'{key}_Match_Field'] = fields
    trials_df[f'{key}_Match_Term'] = terms
    trials_df[f'{key}_Match_Value'] = vals

# Get trial details
print("Fetching trial details...")
details = get_trial_details(set(trials_df['nct_id']))
trials_complete = trials_df.merge(details, on='nct_id', how='left')

# INN in brief or official title: such trials are hidden from an intervention-name search,
# but a full-text search on ClinicalTrials.gov would find them
trials_complete['INN_in_Title'] = [
    inn.lower() in f"{b} {o}".lower()
    for inn, b, o in zip(trials_complete['INN'], trials_complete['brief_title'], trials_complete['official_title'])
]

# Found only via a study-code pattern (patterns/study_code_patterns.csv): every code match came from a pattern
# rule (P1, P2, ...), none from a token match, and the trial is not found via INN
code_rules = (all_matches_df[all_matches_df['search_type'].isin(['Current_Code', 'Legacy_Code'])]
              .groupby(['nct_id', 'INN'])['rule'].agg(set))
pattern_only = {key for key, rules in code_rules.items() if 'token' not in rules}
trials_complete['Found_only_via_pattern'] = [
    (n, i) in pattern_only and not f
    for n, i, f in zip(trials_complete['nct_id'], trials_complete['INN'], trials_complete['Found_via_INN'])
]

# Reorder columns for readability
cols = ['nct_id', 'Brand', 'INN',
        'Found_via_INN', 'INN_Match_Field', 'INN_Match_Term', 'INN_Match_Value',
        'Found_via_Current', 'Current_Match_Field', 'Current_Match_Term', 'Current_Match_Value',
        'Found_via_Legacy', 'Legacy_Match_Field', 'Legacy_Match_Term', 'Legacy_Match_Value',
        'Current_Code', 'Legacy_Codes', 'INN_in_Title', 'Found_only_via_pattern', 'phase', 'start_date', 'overall_status', 'brief_title']
cols = [c for c in cols if c in trials_complete.columns]
trials_complete = trials_complete[cols]

# Deterministic row order: runs can be compared row by row
trials_complete = trials_complete.sort_values(['Brand', 'INN', 'nct_id']).reset_index(drop=True)

print(f"✓ Built verification table: {len(trials_complete)} trials")

# Residue (section 5b): counted = trial already in All_Trials for the same INN (review aid, not fed into counts)
counted_keys = set(zip(trials_complete['nct_id'], trials_complete['INN']))
residue_df['counted'] = [(n, i) in counted_keys for n, i in zip(residue_df['nct_id'], residue_df['INN'])]
not_counted = residue_df[~residue_df['counted']]
print(f"Residue: {len(residue_df)} values, {residue_df['counted'].sum()} in counted trials; "
      f"not counted: {len(not_counted)} values in {not_counted['nct_id'].nunique()} trials")

In [ ]:
# =============================================================================
# HIDDEN TRIALS ANALYSIS
# =============================================================================

hidden = trials_complete[~trials_complete['Found_via_INN']].copy()
print(f"\nHidden trials (not found via INN): {len(hidden)}")
print(f"  of which INN appears in the title: {hidden['INN_in_Title'].sum()}"
      f" — hidden from intervention search only; {(~hidden['INN_in_Title']).sum()} hidden from both")
print(f"  of which found only via a study-code pattern: {hidden['Found_only_via_pattern'].sum()}")

# Breakdown by match field (to see impact of intervention_other_names)
print("\nHidden trials by match field:")
for field in hidden['Current_Match_Field'].dropna().unique():
    if field:
        count = (hidden['Current_Match_Field'].str.contains(field, na=False)).sum()
        print(f"  {field}: {count}")

for field in hidden['Legacy_Match_Field'].dropna().unique():
    if field:
        count = (hidden['Legacy_Match_Field'].str.contains(field, na=False)).sum()
        print(f"  {field}: {count}")

# Display sample with match evidence
print("\nSample hidden trials:")
display_cols = ['nct_id', 'INN', 'Found_via_Current', 'Current_Match_Field', 
                'Current_Match_Term', 'Found_via_Legacy', 'Legacy_Match_Term', 'brief_title']
display(hidden[display_cols].head(15))

## 7. Export Results

In [ ]:
# =============================================================================
# EXPORT TO EXCEL AND CSV
# =============================================================================

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Excel export (multiple sheets)
excel_file = f"{OUTPUT_DIR}/trial_archaeology_{COLLECTION}_{timestamp}.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Summary per drug
    results_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # All trials with match evidence
    trials_complete.to_excel(writer, sheet_name='All_Trials', index=False)
    
    # Hidden trials only (for focused review)
    hidden.to_excel(writer, sheet_name='Hidden_Trials', index=False)

    # Residue: id_information values containing a code key, matched by no rule (human review only)
    residue_df.to_excel(writer, sheet_name='Residue', index=False)
    
    # Statistics
    stats = pd.DataFrame({
        'Metric': [
            'Analysis date', 
            'Collection',
            'AACT last update (studies.updated_at)',
            'Total drugs', 
            'Total unique trials',
            'Trials via INN', 
            'Hidden via current code', 
            'Hidden via legacy',
            'Total hidden', 
            'Pct hidden',
            '',
            'Code fields',
            'Study-code patterns',
        ],
        'Value': [
            datetime.now().strftime('%Y-%m-%d %H:%M'), 
            COLLECTION,
            AACT_LAST_UPDATE,
            len(drugs_df), 
            total_unique, 
            total_inn, 
            total_hidden_current, 
            total_hidden_legacy,
            total_hidden, 
            f"{total_hidden/total_unique*100:.1f}%",
            '',
            '; '.join(label for label, _, _ in CODE_FIELDS),
            '; '.join(p['pattern_id'] for p in PATTERNS),
        ]
    })
    stats.to_excel(writer, sheet_name='Statistics', index=False)

print(f"✓ Exported: {excel_file}")

# CSV export
csv_file = f"{OUTPUT_DIR}/trials_verification_{COLLECTION}_{timestamp}.csv"
trials_complete.to_csv(csv_file, index=False)
print(f"✓ Exported: {csv_file}")

---

## Appendix: Schema Reference

### Tables Used

| Table | Key Field | Purpose |
|-------|-----------|--------|
| `studies` | `nct_id`, `official_title` | Core trial metadata |
| `interventions` | `name` | Primary intervention/drug names |
| `intervention_other_names` | `name` | Aliases, synonyms, development codes |
| `id_information` | `id_value` | Secondary IDs including sponsor protocol codes |

### AACT Documentation

- Schema: https://aact.ctti-clinicaltrials.org/schema
- Data dictionary: https://aact.ctti-clinicaltrials.org/data_dictionary

### Sponsor Code Prefixes

Source: https://en.wikipedia.org/wiki/List_of_pharmaceutical_compound_number_prefixes

| Sponsor | Pattern | Example |
|---------|---------|--------|
| AstraZeneca | AZD + numbers | AZD0914 |
| Bayer | BAY + space + numbers | BAY 3427080 |
| Boehringer Ingelheim | BI + space + numbers | BI 1015550 |
| GSK | GSK + numbers | GSK2140944 |
| Novartis | LOU + numbers | LOU064 |
| Pfizer | PF- + numbers | PF-04554878 |
| Vertex | VX- + numbers | VX-548 |